In [1]:
from llama_cpp import Llama
import json
import pandas as pd
import numpy as np


In [2]:
critic_llm = Llama(
    model_path="../models/tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf",
    n_ctx=1024,
    n_threads=8,
    n_batch=128,
    verbose=True   # ✅ FIX
)

print("✅ LLM Critic loaded")


✅ LLM Critic loaded


AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 0 | SSE3 = 1 | SSSE3 = 0 | VSX = 0 | 


In [3]:
COVE_PROMPT = """
You are a senior data auditor.

Your task:
Verify whether the SQL result correctly answers the question.

Follow these steps internally:
1. Identify what the question is asking (metric, entities, conditions).
2. Identify what the SQL is computing.
3. Compare the SQL result with the question intent.
4. Decide if the answer is semantically correct.

Return ONLY valid JSON in this format:
{{
  "verdict": "correct" | "incorrect",
  "confidence": number between 0 and 1,
  "reason": "short explanation"
}}

Question:
{question}

SQL:
{sql}

Result Preview:
{result}
"""



In [4]:
def summarize_result(df: pd.DataFrame, max_rows: int = 5) -> str:
    if df is None or df.empty:
        return "EMPTY RESULT"

    summary = {
        "columns": list(df.columns),
        "row_count": len(df),
        "sample_rows": df.head(max_rows).to_dict(orient="records")
    }
    return json.dumps(summary, indent=2)


In [5]:
def run_llm_critic(
    question: str,
    sql: str,
    df: pd.DataFrame,
    min_confidence: float = 0.6
):
    result_summary = summarize_result(df)

    prompt = COVE_PROMPT.format(
        question=question,
        sql=sql,
        result=result_summary
    )

    # ✅ CALL THE MODEL, NOT THE FUNCTION
    out = critic_llm(
        prompt,
        max_tokens=256
    )

    text = out["choices"][0]["text"].strip()

    try:
        verdict = json.loads(text)
    except Exception:
        return {
            "ok": False,
            "confidence": 0.0,
            "reason": "Critic output invalid JSON"
        }

    ok = (
        verdict.get("verdict") == "correct"
        and verdict.get("confidence", 0) >= min_confidence
    )

    return {
        "ok": ok,
        "confidence": float(verdict.get("confidence", 0)),
        "reason": verdict.get("reason", "")
    }



In [6]:
def metric_sanity_check(question: str, sql: str) -> bool:
    q = question.lower()
    s = sql.lower()

    if any(k in q for k in ["count", "number of"]) and "count(" not in s:
        return False

    if any(k in q for k in ["average", "avg", "mean"]) and "avg(" not in s:
        return False

    if any(k in q for k in ["maximum", "highest", "max"]) and "max(" not in s:
        return False

    if any(k in q for k in ["minimum", "lowest", "min"]) and "min(" not in s:
        return False

    return True


In [7]:
def semantic_verifier(
    question: str,
    sql: str,
    df: pd.DataFrame,
    min_confidence: float = 0.6
):
    if not metric_sanity_check(question, sql):
        return {
            "ok": False,
            "confidence": 0.0,
            "reason": "Metric mismatch detected"
        }

    return run_llm_critic(
        question=question,
        sql=sql,
        df=df,
        min_confidence=min_confidence
    )


In [9]:
data = {
    "name": ["Alice", "Bob"],
    "title": ["DB Systems", "AI"]
}
df = pd.DataFrame(data)

question = "List student names and course titles"
sql = "SELECT student.name, course.title FROM student JOIN enrollment ON ..."

result = semantic_verifier(question, sql, df)

print(result)


Llama.generate: prefix-match hit


{'ok': False, 'confidence': 0.0, 'reason': 'Critic output invalid JSON'}
